In [1]:
%pip install transformers==4.43.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 104.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 1.0.0rc2
    Uninstalling huggingface-hub-1.0.0rc2:
      Successfully uninstalled huggingface-hub-1.0.0rc2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.2
    Uninstalling tokenizers-0.21.2:
      Successfully uninstalled tokenizers-0.21.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.53.3
    Uninstalling transformers-4.53.3:
      Successfully uninstalled transformers-4.53.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source o

In [1]:
%pip show transformers

Name: transformers
Version: 4.43.4
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: c:\Users\50183\.conda\envs\env_tti\Lib\site-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: peft
Note: you may need to restart the kernel to use updated packages.


In [1]:
from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration
from PIL import Image
import torch

# Load processor and model
model_id = "llava-hf/llava-v1.6-mistral-7b-hf"
processor = LlavaNextProcessor.from_pretrained(model_id,
                                              trust_remote_code=True)
model = LlavaNextForConditionalGeneration.from_pretrained(model_id, 
                                                          torch_dtype=torch.float16, 
                                                          device_map="auto",
                                                         trust_remote_code=True)




Some kwargs in processor config are unused and will not have any effect: vision_feature_select_strategy, patch_size, image_token, num_additional_image_tokens. 
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [4]:
# %pip install transformers -U

In [2]:
def describe_image(image_path,prompt):
    conversation = [
        {
    
          "role": "user",
          "content": [
              {"type": "text", "text": prompt},
              {"type": "image"},
            ],
        },
    ]
    if model.config.pad_token_id is None:
        model.config.pad_token_id = model.config.eos_token_id
        
    with Image.open(image_path) as image:
        prompt = processor.apply_chat_template(conversation, add_generation_prompt=False)
        inputs = processor(text=prompt, images=image, return_tensors="pt").to(model.device)
        with torch.inference_mode():
            output = model.generate(**inputs, 
                                    max_new_tokens=250,
                                    pad_token_id=model.config.pad_token_id,
                                    eos_token_id= model.config.eos_token_id
                                   )
        
        description = processor.decode(output[0], skip_special_tokens=True)
        return description

In [3]:
import re

def remove_prompt(output_text):
    generated_text = re.sub(r"\[INST\](.*?)\[/INST\]", 
                            "", 
                            output_text, 
                            flags=re.DOTALL).strip()
    return generated_text

In [4]:
import os
from tqdm.notebook import tqdm

categories={'dyed-lifted-polyps':'dyed lifted polyps',
           'dyed-resection-margins':'dyed resection margins',
           'esophagitis':'esophagitis',
           'normal-cecum':'normal cecum',
           'normal-pylorus':'normal pylorus',
           'normal-z-line':'normal z-line',
            'polyps':'polyps',
            'ulcerative-colitis':'ulcerative colitis'
           }
data_root = r'C:\COMP9800\Dataset\kvasir-dataset'
category_stats = {}
image_desc = []

for folder, category in tqdm(categories.items(), desc='Describe Images'):
    folder_path = data_root + '/' + folder
    images =  os.listdir(folder_path)
    prompt = f"An endoscopic image of {category}. Please describe the endoscopic image of {category} using the following visual features: specific color, shape, surrounding mucosa, surface texture, bleeding condition, and other relevant details. Start with ‘An endoscopic image of {category}’."
    print(f'Current Category: {folder}')
    for img in tqdm(images[:50], desc=folder):
        img_path = folder_path + '/' + img
        desc = describe_image(image_path=img_path, prompt=prompt)

        image_desc.append({'category':folder, 'img': img, 'desc':remove_prompt(desc)})
        break

    # category_stats[folder] = len(images)

# print(category_stats)

Describe Images:   0%|          | 0/8 [00:00<?, ?it/s]

Current Category: dyed-lifted-polyps


dyed-lifted-polyps:   0%|          | 0/50 [00:00<?, ?it/s]

We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


KeyboardInterrupt: 

In [8]:
import csv

file_name = 'image_description.csv'
with open(file_name,mode='w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f,fieldnames=['category','img','desc'], quoting=csv.QUOTE_ALL)
    writer.writeheader()
    for record in image_desc:
        writer.writerow(record)

## Prepare Dataset for Compressor

In [4]:
import pandas as pd

image_desc_file = 'C:\COMP9800\Dataset\kvasir-dataset\image_description_partial.csv'

data = pd.read_csv(image_desc_file)
description = data.iloc[:,2]

In [15]:
help(summarizer)

Help on SummarizationPipeline in module transformers.pipelines.text2text_generation object:

class SummarizationPipeline(Text2TextGenerationPipeline)
 |  SummarizationPipeline(*args, **kwargs)
 |  
 |  Summarize news articles and other documents.
 |  
 |  This summarizing pipeline can currently be loaded from [`pipeline`] using the following task identifier:
 |  `"summarization"`.
 |  
 |  The models that this pipeline can use are models that have been fine-tuned on a summarization task, which is
 |  currently, '*bart-large-cnn*', '*google-t5/t5-small*', '*google-t5/t5-base*', '*google-t5/t5-large*', '*google-t5/t5-3b*', '*google-t5/t5-11b*'. See the up-to-date
 |  list of available models on [huggingface.co/models](https://huggingface.co/models?filter=summarization). For a list
 |  of available parameters, see the [following
 |  documentation](https://huggingface.co/docs/transformers/en/main_classes/text_generation#transformers.generation.GenerationMixin.generate)
 |  
 |  Usage:
 |  

In [1]:
from transformers import pipeline

summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

In [19]:
long_text = description.to_list()
results=[]
for t in long_text:
    summ = summarizer(t, max_length=75, min_length=60, do_sample=False)
    results.append(summ)



In [23]:
description

0      An endoscopic image of dyed lifted polyps show...
1      An endoscopic image of dyed lifted polyps is d...
2      An endoscopic image of dyed lifted polyps typi...
3      An endoscopic image of dyed lifted polyps show...
4      An endoscopic image of dyed lifted polyps is d...
                             ...                        
395    An endoscopic image of ulcerative colitis typi...
396    An endoscopic image of ulcerative colitis typi...
397    An endoscopic image of ulcerative colitis typi...
398    An endoscopic image of ulcerative colitis typi...
399    An endoscopic image of ulcerative colitis typi...
Name: desc, Length: 400, dtype: object

In [24]:
compressor_df = pd.DataFrame({'long_text':description,'short_text':[r[0]['summary_text'] for r in results]})

In [27]:
compressor_df.to_csv(r'C:\COMP9800\Dataset\text_compressor_ds.csv')